# Shri Yogavasishtha — Chandra OCR
**Runtime:** GPU → T4 (Runtime > Change runtime type)

OCR runs to **local Colab storage** first (fast, reliable), then syncs to Google Drive at the end.

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────────
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print(r.stdout.strip() if r.returncode == 0 else 'NO GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── Cell 2: Install chandra-ocr ─────────────────────────────────────────────
!pip install -q chandra-ocr[hf]
import subprocess
print('chandra:', subprocess.run(['which','chandra'], capture_output=True, text=True).stdout.strip())

In [ ]:
# ── Cell 3: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# Local NVMe — fast, reliable for OCR writes
LOCAL_DIR = '/content/yogavasishtha_ocr'
# Google Drive — final destination
DRIVE_DIR = '/content/drive/MyDrive/yogavasishtha_ocr'

os.makedirs(LOCAL_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Local:  {LOCAL_DIR}')
print(f'Drive:  {DRIVE_DIR}')

In [ ]:
# ── Cell 4: Upload PDFs ─────────────────────────────────────────────────────
from google.colab import files as colab_files
import shutil

print('Select all 4 PDFs (Shri-Yogavasishtha-1.pdf through 4.pdf)')
uploaded = colab_files.upload()

pdf_paths = []
for filename in uploaded:
    dest = f'/content/{filename}'
    if not os.path.exists(dest):
        shutil.move(filename, dest)
    pdf_paths.append(dest)
    print(f'  {filename}: {os.path.getsize(dest)/1e6:.1f} MB')

pdf_paths.sort()
print(f'\nReady: {[os.path.basename(p) for p in pdf_paths]}')

In [ ]:
# ── Cell 4-alt: PDFs already in Drive ───────────────────────────────────────
# Only run this if PDFs are already in Drive (skip Cell 4)
PDF_DRIVE_DIR = '/content/drive/MyDrive/Yogavasishtha_PDFs'  # change if needed
pdf_paths = sorted([os.path.join(PDF_DRIVE_DIR, f)
                    for f in os.listdir(PDF_DRIVE_DIR) if f.endswith('.pdf')])
for p in pdf_paths: print(p)

In [ ]:
# ── Cell 5: Smoke test — OCR 2 pages, see what chandra creates ──────────────
import time, os

CHANDRA = subprocess.run(['which','chandra'], capture_output=True, text=True).stdout.strip()
test_out = '/content/smoke_test'

# Fix CUDA OOM: expandable segments + batch-size 1
env = os.environ.copy()
env['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

print('Running smoke test (2 pages)...')
t0 = time.time()
r = subprocess.run(
    [CHANDRA, pdf_paths[0], test_out,
     '--method', 'hf',
     '--page-range', '1-2',
     '--batch-size', '1',    # 1 page at a time → prevents CUDA OOM
     '--max-workers', '1'],  # 1 worker → keeps VRAM flat
    capture_output=True, text=True, env=env
)
print(f'Return code: {r.returncode}  ({time.time()-t0:.1f}s)')
if r.stderr: print('STDERR (last 800 chars):', r.stderr[-800:])

# Show only files chandra actually created (skip PDFs and system dirs)
skip = {'/content/sample_data', '/content/.config'}
print('\nFiles created by chandra:')
found = False
for root, dirs, fnames in os.walk('/content'):
    if any(root.startswith(d) for d in skip) or '/drive' in root: continue
    for f in fnames:
        fp = os.path.join(root, f)
        if fp.endswith('.pdf'): continue
        print(f'  {fp}  ({os.path.getsize(fp):,} bytes)')
        found = True
if not found:
    print('  (none — check STDERR above)')

In [ ]:
# ── Cell 6: Preview smoke test content ──────────────────────────────────────
import re

def find_text_files(path):
    """Return sorted list of .md/.txt files at path (file or dir)."""
    if os.path.isfile(path) and os.path.splitext(path)[1] in ('.md','.txt'):
        return [path]
    out = []
    if os.path.isdir(path):
        for root, dirs, fnames in os.walk(path):
            for f in fnames:
                if f.endswith(('.md','.txt')):
                    out.append(os.path.join(root, f))
    return sorted(out)

# Try the test output path with and without extension
files_found = find_text_files('/content/smoke_test') or \
              find_text_files('/content/smoke_test.md')

if files_found:
    print(f'Found {len(files_found)} output file(s):')
    for f in files_found:
        print(f'  {f}  ({os.path.getsize(f):,} bytes)')
    print('\n--- Content preview ---')
    with open(files_found[0], 'r', errors='replace') as f:
        print(f.read(1500))
else:
    print('No output files found — check Cell 5 errors')

In [ ]:
# ── Cell 7: Run OCR on all 4 PDFs → local storage ───────────────────────────
import re

def find_text_files(path):
    if os.path.isfile(path) and os.path.splitext(path)[1] in ('.md','.txt'):
        return [path]
    out = []
    if os.path.isdir(path):
        for root, dirs, fnames in os.walk(path):
            for f in fnames:
                if f.endswith(('.md','.txt')):
                    out.append(os.path.join(root, f))
    return sorted(out)

env = os.environ.copy()
env['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

print(f'chandra: {CHANDRA}\n')

for pdf_path in pdf_paths:
    name = os.path.splitext(os.path.basename(pdf_path))[0]
    out_arg = os.path.join(LOCAL_DIR, f'{name}.md')

    existing = find_text_files(out_arg)
    if existing:
        kb = sum(os.path.getsize(f) for f in existing) / 1024
        print(f'SKIP {name} — already exists ({kb:.0f} KB)')
        continue

    print(f'\n━━━ {name} ━━━')
    t0 = time.time()

    proc = subprocess.run([
        CHANDRA, pdf_path, out_arg,
        '--method', 'hf',
        '--no-headers-footers',
        '--batch-size', '1',    # prevents CUDA OOM on T4
        '--max-workers', '1',
    ], text=True, env=env)

    elapsed = time.time() - t0
    created = find_text_files(out_arg)
    kb = sum(os.path.getsize(f) for f in created) / 1024 if created else 0
    status = 'DONE' if proc.returncode == 0 else f'ERROR({proc.returncode})'
    print(f'{status} — {elapsed/60:.1f} min | {len(created)} file(s) | {kb:.0f} KB')

print('\n✓ OCR complete')

In [ ]:
# ── Cell 8: Merge pages + copy to Drive ─────────────────────────────────────
import shutil

def page_num(path):
    nums = re.findall(r'\d+', os.path.basename(path))
    return int(nums[-1]) if nums else 0

for pdf_path in pdf_paths:
    name = os.path.splitext(os.path.basename(pdf_path))[0]

    # Collect output files (try both single-file and dir patterns)
    page_files = find_text_files(os.path.join(LOCAL_DIR, f'{name}.md')) or \
                 find_text_files(os.path.join(LOCAL_DIR, name))

    if not page_files:
        print(f'SKIP {name} — no output files found locally')
        continue

    page_files = sorted(page_files, key=page_num)

    if len(page_files) == 1:
        merged_local = page_files[0]
    else:
        merged_local = os.path.join(LOCAL_DIR, f'{name}_merged.md')
        with open(merged_local, 'w', encoding='utf-8') as out_f:
            out_f.write(f'# {name}\n\n')
            for pf in page_files:
                pn = page_num(pf)
                out_f.write(f'\n---\n<!-- page {pn} -->\n\n')
                with open(pf, 'r', encoding='utf-8', errors='replace') as f:
                    out_f.write(f.read())

    drive_dest = os.path.join(DRIVE_DIR, f'{name}_merged.md')
    shutil.copy2(merged_local, drive_dest)
    kb = os.path.getsize(drive_dest) / 1024
    print(f'{name}: {len(page_files)} page file(s) → Drive ({kb:.0f} KB) ✓')

print(f'\n✓ All synced to {DRIVE_DIR}')

In [ ]:
# ── Cell 9: Preview Part 1 ───────────────────────────────────────────────────
candidates = sorted([f for f in os.listdir(DRIVE_DIR) if f.endswith('_merged.md')])
if candidates:
    with open(os.path.join(DRIVE_DIR, candidates[0]), 'r', encoding='utf-8') as f:
        print(f.read(2000))
else:
    print('No merged files on Drive yet — run Cells 7 and 8 first')

In [ ]:
# ── Cell 10 (optional): Download all merged files as a zip ──────────────────
import zipfile
from google.colab import files as colab_files

zip_path = '/content/yogavasishtha_ocr.zip'
merged = [f for f in os.listdir(DRIVE_DIR) if f.endswith('_merged.md')]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in merged:
        zf.write(os.path.join(DRIVE_DIR, fname), fname)

print(f'Zip: {os.path.getsize(zip_path)/1e6:.1f} MB  |  {merged}')
colab_files.download(zip_path)